# IAN 620 · Module 9
## End-to-End Agentic RAG with a Downloadable Local LLM, LangChain, Chroma, DuckDuckGo, and LangGraph

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ali2449/IAN-620/blob/main/IAN_620_Module_9_Agentic_RAG_Labs.ipynb)

This notebook builds one complete workflow from the ground up. **No paid LLM API is required for the core notebook.** A small open model is downloaded from Hugging Face and runs directly in the notebook.

**Learning path**

1. Understand Agentic AI, RAG, Agentic RAG, LangChain, and LangGraph.
2. Load and inspect a small knowledge-base dataset.
3. Build Documents, chunks, local embeddings, Chroma, and a retriever.
4. Download and run a local Qwen3 language model.
5. Build classic RAG.
6. Turn retrieval into a LangChain tool and add DuckDuckGo.
7. **Design the project before the agent:** choose architecture, define tool contracts, manage a tool registry, and refine input.
8. Build a bounded plain-Python Agentic RAG loop with no-progress detection.
9. Add a visible evaluator → revision loop using draft, feedback, and revision budget.
10. Optionally compare with LangChain `create_agent`.
11. Build the same bounded workflow with LangGraph, including an explicit safe-stop branch.
12. Reload Chroma and evaluate retrieval behavior.

> Default local model: `Qwen/Qwen3-0.6B`. It downloads automatically from Hugging Face, requires no API key, and can be replaced with a larger local model later.

### Architecture used in this notebook

| Layer | What we use | Why |
|---|---|---|
| Dataset | CSV | Easy to inspect and reuse |
| Documents | `Document` | Standard LangChain data object |
| Chunking | `RecursiveCharacterTextSplitter` | Simple general-purpose chunking |
| Embeddings | `all-MiniLM-L6-v2` | Small local embedding model |
| Vector database | Chroma | Persistent local vector store |
| Retrieval | `as_retriever()` | Standard retriever interface |
| Local LLM | `Qwen/Qwen3-0.6B` | Downloadable; no API key required |
| RAG | Retriever + prompt + local LLM | Grounded generation |
| Tools | Retriever tool + DuckDuckGo | Internal knowledge + external web |
| Agent | Plain Python + LangGraph | Explicit decisions and bounded loops |
| Optional comparison | LangChain `create_agent` | High-level provider/tool-calling example |

## 0. Before the code: what is Agentic AI?

An **LLM application** sends input to a model and receives output. An **agentic AI system** adds a control loop around the model so the system can maintain state, choose among approved actions, observe results, update its state, and continue until a stop condition is reached.

That means the model is only one component. The agentic behavior comes from the surrounding software.

![Agentic AI control loop](https://raw.githubusercontent.com/ali2449/IAN-620/main/src/img/module9_agentic_ai_foundation.svg)

### The five pieces of an agentic system

| Piece | Question it answers | Example in this notebook |
|---|---|---|
| **Goal** | What is the user trying to accomplish? | Answer a support-policy question |
| **State** | What does the workflow know right now? | Question, query, retrieved documents, score, step count |
| **Decision** | What should happen next? | Retrieve, rewrite, search the web, generate, or stop |
| **Action / tool** | What can the system do? | Chroma retrieval, DuckDuckGo search, local LLM generation |
| **Stop condition** | When should the workflow end? | Enough evidence, maximum steps, safe failure, or final answer |

### LLM, workflow, agent, RAG, and Agentic RAG are not the same thing

| System | What it does | Does it choose the next action? |
|---|---|---:|
| **LLM call** | prompt → model → response | No |
| **Deterministic workflow** | fixed sequence of programmed steps | Usually no |
| **RAG** | retrieve → add context → generate | Usually no |
| **Agent** | maintains state and chooses among approved actions | Yes |
| **Agentic RAG** | makes retrieval part of the agent's decision loop | Yes |

A fixed workflow can still contain an LLM. An agent can still contain deterministic rules. In a good design, we use normal code for predictable decisions and model-driven decisions only where flexible language judgment is useful.

### The core agent loop in plain Python

Before using any framework, reduce the idea to normal control flow:

```python
state = make_state(user_request)

while not finished(state):
    action = choose_next_action(state)
    observation = run_action(action)
    state = update_state(state, observation)

return final_result(state)
```

Agent frameworks do not remove this logic. They give us reusable structures for representing it.

## Why use LangChain at all?

You **do not need LangChain** to build RAG or an agent. Plain Python is often the best place to start because every function call and branch is visible.

LangChain becomes useful when an application starts combining many reusable AI components. It provides common interfaces and integrations for:

- chat models from different providers or local runtimes,
- `Document` objects and document loaders,
- text splitters,
- embeddings and vector stores,
- retrievers,
- tools,
- prompts and messages,
- higher-level agent harnesses such as `create_agent`.

The practical benefit is that your controller can work with a **retriever interface** or a **tool interface** instead of being tightly coupled to one vector database or one model vendor.

Current reference: [LangChain agents documentation](https://docs.langchain.com/oss/python/langchain/agents).

## Why use LangGraph?

LangGraph is a lower-level orchestration framework for **stateful, multi-step agents and workflows**. It is useful when the control flow itself has become important.

Use LangGraph when you need features such as:

- explicit shared state,
- branching and conditional edges,
- loops and retries,
- a mixture of deterministic code and LLM-driven decisions,
- persistence / checkpoints,
- long-running execution,
- human-in-the-loop approval,
- inspectable execution paths.

LangGraph does **not** replace the LLM, retriever, Chroma, or tools. It orchestrates them.

Also, LangGraph can be used without LangChain. In practice, LangChain components are convenient for models, retrievers, and tools, while LangGraph controls how those components are connected.

Current reference: [LangGraph overview](https://docs.langchain.com/oss/python/langgraph/overview).

## How the pieces fit together

![LangChain and LangGraph stack](https://raw.githubusercontent.com/ali2449/IAN-620/main/src/img/module9_langchain_langgraph_stack.svg)

| Component | Job in this notebook |
|---|---|
| **Qwen3 local LLM** | Generates answers; no API key required |
| **Chroma** | Stores embedding vectors and retrieves similar chunks |
| **DuckDuckGo** | Provides optional public-web search |
| **LangChain** | Gives us `Document`, retriever, tool, and model integration patterns |
| **LangGraph** | Controls state, routing, grading, rewriting, loops, and stopping |

Think of LangChain as the **component/integration layer** and LangGraph as the **orchestration/control-flow layer**.

## What is Agentic RAG?

Classic RAG normally follows a fixed sequence:

**question → retrieve → context → LLM → answer**

Agentic RAG puts retrieval inside a decision loop. The system can decide:

- whether retrieval is needed,
- which source or tool should be used,
- whether the retrieved evidence is good enough,
- whether the query should be rewritten,
- whether another source should be tried,
- whether to generate an answer or stop safely.

![Classic RAG versus Agentic RAG](https://raw.githubusercontent.com/ali2449/IAN-620/main/src/img/module9_rag_to_agentic_rag.svg)

Current reference: [LangGraph custom RAG agent tutorial](https://docs.langchain.com/oss/python/langgraph/agentic-rag).

### Classic RAG or Agentic RAG?

| Situation | Better starting point | Why |
|---|---|---|
| One stable knowledge base and one retrieval step | **Classic RAG** | Simpler, faster, cheaper |
| Retrieval sometimes fails and needs rewriting | **Agentic RAG** | Can inspect and correct retrieval |
| Question may require vector search, SQL, API, or web | **Agentic RAG** | Can route to different tools |
| Workflow is completely predictable | **Plain Python workflow** | No reason to ask an LLM to choose |
| High-impact action such as refund approval | **Bounded agent + deterministic rules + human approval** | Model should not own authorization |

Agentic RAG should be used because the problem needs conditional retrieval behavior—not because agents are newer or more sophisticated.

## Architecture we will build

The notebook intentionally grows the system in layers:

1. **Data layer:** CSV → LangChain `Document` objects → chunks.
2. **Representation layer:** local embeddings.
3. **Retrieval layer:** Chroma → retriever.
4. **Generation layer:** downloadable Qwen3 local LLM.
5. **Classic RAG:** fixed retrieve → generate pipeline.
6. **Tool layer:** retriever tool + DuckDuckGo.
7. **Agentic layer:** state + grading + rewriting + routing.
8. **LangGraph layer:** the same controller expressed as nodes and conditional edges.
9. **Evaluation layer:** inspect retrieval results, trajectory, and stopping behavior.

The most important goal is not memorizing framework syntax. It is being able to explain **why each component exists and what would happen without it**.

## 1. Install the packages

Run this cell once in Colab. After installation, Colab may ask you to restart the runtime if a package was already loaded.

In [ ]:
!pip -q install -U \
    langchain \
    langgraph \
    langchain-chroma \
    langchain-community \
    langchain-huggingface \
    langchain-openai \
    chromadb \
    "sentence-transformers>=5.2.0" \
    "transformers>=4.52.4" \
    accelerate \
    duckduckgo-search

## 2. Imports and version check

Keeping the imports together makes it easier to see which library owns each part of the workflow.

In [ ]:
import os
import shutil
from pathlib import Path

import pandas as pd

import langchain
import langgraph
import chromadb

print('LangChain:', langchain.__version__)
print('LangGraph:', getattr(langgraph, '__version__', 'installed'))
print('Chroma:', chromadb.__version__)

## 3. Load the dataset

The CSV is stored in the same GitHub repository as this notebook. In Colab, we read it directly from the raw GitHub URL.

In [ ]:
DATA_URL = "https://raw.githubusercontent.com/ali2449/IAN-620/main/IAN_620_Module_9_Agentic_RAG_Knowledge_Base.csv"

kb = pd.read_csv(DATA_URL)
print('Rows:', len(kb))
print('Columns:', list(kb.columns))
kb.head()

### Inspect the knowledge base

Each row is a short policy or support document. The `source_id` gives us a stable citation identifier, and the metadata fields can later be used for filtering.

In [ ]:
kb[['source_id', 'category', 'title', 'effective_date']].head(10)

In [ ]:
kb['category'].value_counts()

## 4. Convert rows into LangChain documents

A `Document` keeps the text together with metadata. This is important because a RAG system should preserve where each piece of evidence came from.

In [ ]:
from langchain_core.documents import Document

documents = []

for _, row in kb.iterrows():
    documents.append(
        Document(
            page_content=f"{row['title']}\n{row['text']}",
            metadata={
                'source_id': row['source_id'],
                'category': row['category'],
                'title': row['title'],
                'effective_date': row['effective_date'],
                'source_url': row['source_url'],
            },
        )
    )

print('Documents:', len(documents))
print(documents[0])

## 5. Chunk the documents

The example rows are short, but we still use a text splitter so the notebook follows the same workflow used for longer documents.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=350,
    chunk_overlap=50,
)

chunks = splitter.split_documents(documents)

print('Original documents:', len(documents))
print('Chunks:', len(chunks))
print(chunks[0].page_content)

## 6. Create local embeddings

`all-MiniLM-L6-v2` runs locally and is small enough for a teaching notebook. Embeddings turn each chunk into a numeric vector so semantically similar text can be retrieved.

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

embeddings = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL,
    encode_kwargs={'normalize_embeddings': True},
)

sample_vector = embeddings.embed_query('return policy after delivery')
print('Embedding dimensions:', len(sample_vector))
print('First 5 values:', sample_vector[:5])

## 7. Build a persistent Chroma vector database

Chroma stores the chunk vectors and metadata. `persist_directory` lets us reload the same collection later instead of rebuilding it every time.

In [ ]:
from langchain_chroma import Chroma

CHROMA_DIR = "./ian620_module9_chroma"
COLLECTION_NAME = "ian620_module9_support"

# Start clean so re-running the notebook does not duplicate the teaching data.
shutil.rmtree(CHROMA_DIR, ignore_errors=True)

vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    ids=[f'chunk-{i:03d}' for i in range(len(chunks))],
    collection_name=COLLECTION_NAME,
    persist_directory=CHROMA_DIR,
)

print('Chroma directory:', Path(CHROMA_DIR).resolve())
print('Stored chunks:', len(vector_store.get()['ids']))

## 8. Test semantic retrieval

Start by calling the vector store directly. This keeps retrieval separate from generation so we can inspect whether the correct evidence is being found.

In [ ]:
query = "How long do I have to return a normal item after delivery?"

results = vector_store.similarity_search(query, k=3)

for i, doc in enumerate(results, start=1):
    print(f'--- Result {i} ---')
    print('Source:', doc.metadata['source_id'])
    print('Category:', doc.metadata['category'])
    print(doc.page_content)
    print()

## 9. Create a retriever

A retriever gives us a simple interface: give it a question and receive the most relevant documents.

In [ ]:
retriever = vector_store.as_retriever(
    search_kwargs={'k': 3}
)

docs = retriever.invoke(
    "When will a refund reach the original payment method?"
)

[(d.metadata['source_id'], d.metadata['title']) for d in docs]

## 10. Download and run a local LLM — no API key

The notebook uses **Qwen3-0.6B** as the default generator. The model is downloaded automatically from Hugging Face the first time this cell runs.

- No OpenAI, Anthropic, Gemini, or other LLM API key is required.
- The standard Transformers checkpoint is about 1.5 GB.
- A GPU is recommended in Colab: **Runtime → Change runtime type → GPU**.
- CPU also works, but generation will be slower.
- For a stronger machine, you can later change the model name to `Qwen/Qwen3-1.7B`.

The model files are cached locally by Hugging Face, so later runs can reuse the downloaded model when the cache is still available.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

LOCAL_MODEL_NAME = "Qwen/Qwen3-0.6B"

tokenizer = AutoTokenizer.from_pretrained(
    LOCAL_MODEL_NAME
)

local_model = AutoModelForCausalLM.from_pretrained(
    LOCAL_MODEL_NAME,
    torch_dtype="auto",
    device_map="auto",
)

print('Local model:', LOCAL_MODEL_NAME)
print('Device:', local_model.device)

In [ ]:
def local_generate(prompt, max_new_tokens=220):
    messages = [
        {
            'role': 'system',
            'content': (
                'You are a concise support assistant. '
                'Use supplied evidence carefully and do not invent facts.'
            ),
        },
        {'role': 'user', 'content': prompt},
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    ).to(local_model.device)

    with torch.no_grad():
        outputs = local_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
        )

    new_tokens = outputs[0][inputs['input_ids'].shape[-1]:]

    return tokenizer.decode(
        new_tokens,
        skip_special_tokens=True,
    ).strip()


print(local_generate(
    'In one sentence, explain what retrieval-augmented generation is.',
    max_new_tokens=80,
))

## 11. Build classic RAG with the local model

Classic RAG follows a fixed path: **question → retrieve → context → local LLM → answer**. There is no correction loop yet, and no API key is needed.

In [ ]:
def format_context(docs):
    parts = []
    for doc in docs:
        sid = doc.metadata.get('source_id', 'unknown')
        parts.append(f'[{sid}] {doc.page_content}')
    return '\n\n'.join(parts)


def classic_rag(question):
    docs = retriever.invoke(question)
    context = format_context(docs)

    prompt = f'''
Answer the question using only the supplied context.
If the context is insufficient, say that you cannot verify the answer.
Cite source IDs such as [KB001].

CONTEXT:
{context}

QUESTION:
{question}
'''

    answer = local_generate(prompt)

    return {
        'answer': answer,
        'context': context,
        'documents': docs,
    }

In [ ]:
question = "Can I return a standard item 20 days after delivery?"
rag_result = classic_rag(question)

print(rag_result['answer'])
print('\nRetrieved sources:')
print([d.metadata['source_id'] for d in rag_result['documents']])

## 12. Turn the retriever into a LangChain tool

A retriever tool is the bridge from RAG to an agent. The model can now request retrieval as an action instead of the application always retrieving automatically.

In [ ]:
from langchain_core.tools import create_retriever_tool

retriever_tool = create_retriever_tool(
    retriever,
    name='search_support_knowledge',
    description=(
        'Search the approved internal support knowledge base for '
        'returns, refunds, shipping, warranty, accounts, billing, '
        'subscriptions, privacy, support hours, and related policies.'
    ),
)

print('Tool name:', retriever_tool.name)
print('Tool description:', retriever_tool.description)

In [ ]:
print(
    retriever_tool.invoke(
        {"query": "What is the standard return deadline?"}
    )
)

## 13. Add DuckDuckGo web search

DuckDuckGo gives the agent an external search tool that does not require a separate API key. Keep internal knowledge and public-web evidence conceptually separate.

**Policy used in this notebook:** internal company questions should use the knowledge-base retriever. Web search is for explicitly external or current information.

In [ ]:
from langchain_community.tools import DuckDuckGoSearchRun

web_search = DuckDuckGoSearchRun()

try:
    web_result = web_search.invoke(
        "LangGraph StateGraph official documentation"
    )
    print(str(web_result)[:1200])
except Exception as exc:
    print('Web search could not run in this environment:')
    print(exc)

## 14. Design the project before building the agent

Before adding an agent loop, decide **why** the project needs one.

| Situation | Start with |
|---|---|
| Every step is known | Plain Python / fixed workflow |
| Need answers from internal documents | Classic RAG |
| Several approved tools may be needed | Agent |
| Retrieval may need grading, rewriting, or multiple sources | Agentic RAG |
| Need explicit state, loops, persistence, or approval | LangGraph |

### Tool selection rule

- **Exact rule / calculation** → Python function.
- **Internal unstructured knowledge** → retriever / RAG.
- **Exact structured fact** → SQL / API tool.
- **Current public information** → web-search tool.
- **Real-world side effect** → action tool + authorization + often human approval.

A strong project does not maximize the number of agents. It gives every model call and every tool a clear job.

### 14.1 Give each tool a narrow contract

Each tool should have:

- a specific verb-object name,
- a description that says when to use it and when not to use it,
- a small validated input schema,
- predictable output with provenance where appropriate,
- explicit read/write permissions,
- timeout/retry/approval policy.

Below we add a structured order lookup so the project has both **unstructured retrieval** and an **exact structured-data tool**.

In [ ]:
from langchain_core.tools import tool

ORDER_DB = {
    '1042': {
        'status': 'delivered',
        'delivered_days_ago': 20,
        'item_condition': 'unused',
    },
    '1043': {
        'status': 'in_transit',
        'delivered_days_ago': None,
        'item_condition': 'unknown',
    },
}


@tool
def get_order(order_id: str) -> dict:
    """Get exact structured facts for one order.
Use this for order status or delivery information.
Do not use it for return-policy rules."""

    order = ORDER_DB.get(order_id)

    if order is None:
        return {
            'found': False,
            'order_id': order_id,
        }

    return {
        'found': True,
        'order_id': order_id,
        **order,
    }


print(get_order.invoke({'order_id': '1042'}))

### 14.2 Manage tools with a registry

A registry keeps tool metadata outside the model. The controller can use it to expose only tools appropriate for the current task.

Useful metadata includes:

- tool kind,
- read-only vs. action,
- rough cost/latency,
- whether human approval is required.

Permissions belong in application code—not only in the system prompt.

In [ ]:
TOOL_REGISTRY = {
    retriever_tool.name: {
        'tool': retriever_tool,
        'kind': 'knowledge',
        'read_only': True,
        'cost': 'low',
        'approval': False,
    },
    get_order.name: {
        'tool': get_order,
        'kind': 'database',
        'read_only': True,
        'cost': 'low',
        'approval': False,
    },
    web_search.name: {
        'tool': web_search,
        'kind': 'web',
        'read_only': True,
        'cost': 'medium',
        'approval': False,
    },
}

pd.DataFrame([
    {
        'name': name,
        **{k: v for k, v in meta.items() if k != 'tool'},
    }
    for name, meta in TOOL_REGISTRY.items()
])

### 14.3 Expose only relevant tools

Do not automatically give every agent every tool.

A smaller, relevant tool set reduces routing ambiguity and makes permissions easier to reason about.

In [ ]:
def tools_for_project(
    needs_order=False,
    allow_web=False,
):
    names = [retriever_tool.name]

    if needs_order:
        names.append(get_order.name)

    if allow_web:
        names.append(web_search.name)

    return [
        TOOL_REGISTRY[name]['tool']
        for name in names
    ]


selected_tools = tools_for_project(
    needs_order=True,
    allow_web=False,
)

print([tool.name for tool in selected_tools])

### 14.4 Refine the input without changing the user's intent

Input refinement should produce **visible application artifacts**, not hidden chain-of-thought.

Keep the original question. Create separate fields for:

- normalized/refined search query,
- extracted order ID,
- whether structured order data is needed,
- whether public web search is explicitly allowed.

This makes later routing testable.

In [ ]:
import re


def refine_input(question: str):
    normalized = ' '.join(question.strip().split())

    order_match = re.search(
        r'\border\s*#?\s*(\d+)\b',
        normalized,
        flags=re.IGNORECASE,
    )

    order_id = (
        order_match.group(1)
        if order_match
        else None
    )

    q = normalized.lower()
    needs_order = order_id is not None
    needs_web = any(
        word in q
        for word in [
            'latest',
            'current web',
            'public web',
            'internet',
        ]
    )

    return {
        'question': question,
        'refined_query': normalized,
        'order_id': order_id,
        'needs_order': needs_order,
        'needs_web': needs_web,
    }


refined = refine_input(
    "Can I return order 1042 after 20 days?"
)

refined

### 14.5 Build a small visible plan

A plan should describe **what information is needed and which approved capabilities can provide it**.

It does not need to contain hidden reasoning.

Example project plan:

```text
Goal: answer return eligibility.
Need: return policy + order delivery fact.
Tools: internal retriever + get_order.
Stop: enough evidence or maximum attempts reached.
```

In [ ]:
def make_plan(refined_input):
    tools = [retriever_tool.name]

    if refined_input['needs_order']:
        tools.append(get_order.name)

    if refined_input['needs_web']:
        tools.append(web_search.name)

    return {
        'goal': 'answer the user from approved evidence',
        'tools': tools,
        'max_retrieval_steps': 3,
    }


plan = make_plan(refined)
plan

## 15. A bounded plain-Python Agentic RAG loop

Before LangChain agents or LangGraph, make the controller explicit.

This version:

- preserves the original question,
- retrieves evidence,
- grades evidence,
- rewrites only when needed,
- detects repeated evidence/no progress,
- stops safely when evidence is still weak,
- generates only when the evidence threshold is met.

In [ ]:
def evidence_score(question, docs):
    if not docs:
        return 0.0

    q_words = {
        w.strip('.,?!').lower()
        for w in question.split()
        if len(w) > 3
    }

    text = ' '.join(
        d.page_content.lower()
        for d in docs
    )

    if not q_words:
        return 0.0

    matched = sum(
        word in text
        for word in q_words
    )

    return matched / len(q_words)


def rewrite_query(question):
    q = question.lower()

    if 'return' in q:
        return (
            question
            + ' return window delivery deadline refund policy'
        )

    if 'refund' in q:
        return (
            question
            + ' refund processing payment method business days'
        )

    if 'shipping' in q or 'delivery' in q:
        return (
            question
            + ' shipping delivery tracking policy'
        )

    return (
        question
        + ' support policy requirements eligibility'
    )

In [ ]:
def run_plain_agentic_rag(
    question,
    max_steps=3,
    threshold=0.25,
):
    refined = refine_input(question)
    query = refined['refined_query']

    docs = []
    score = 0.0
    trace = []
    previous_sources = None
    stop_reason = 'max_steps'

    for step in range(1, max_steps + 1):
        docs = retriever.invoke(query)

        sources = tuple(
            d.metadata.get('source_id', 'unknown')
            for d in docs
        )

        score = evidence_score(
            question,
            docs,
        )

        trace.append({
            'step': step,
            'action': 'retrieve_and_grade',
            'query': query,
            'score': round(score, 3),
            'sources': list(sources),
        })

        if score >= threshold:
            stop_reason = 'evidence_sufficient'
            break

        if (
            previous_sources is not None
            and sources == previous_sources
        ):
            stop_reason = 'no_progress_same_evidence'
            trace.append({
                'step': step,
                'action': 'stop',
                'reason': stop_reason,
            })
            break

        new_query = rewrite_query(question)

        if new_query == query:
            stop_reason = 'no_progress_same_query'
            trace.append({
                'step': step,
                'action': 'stop',
                'reason': stop_reason,
            })
            break

        trace.append({
            'step': step,
            'action': 'rewrite',
            'query': new_query,
        })

        previous_sources = sources
        query = new_query

    if score < threshold:
        answer = (
            'I could not find enough reliable evidence '
            'to answer this question.'
        )
    else:
        context = format_context(docs)

        prompt = f'''
Use only the verified evidence below.
If evidence is insufficient, say so.
Cite source IDs.

EVIDENCE:
{context}

QUESTION:
{question}
'''

        answer = local_generate(prompt)

    return {
        'answer': answer,
        'documents': docs,
        'score': score,
        'stop_reason': stop_reason,
        'trace': trace,
    }

In [ ]:
plain_result = run_plain_agentic_rag("Can a normal item be returned after 20 days?")

print(plain_result['answer'])
print('Stop reason:', plain_result['stop_reason'])
print('Score:', round(plain_result['score'], 3))

pd.DataFrame(plain_result['trace'])

## 16. Add deliberate refinement with visible artifacts

“More thinking” should mean **more explicit, testable passes**, not hidden chain-of-thought.

A useful pattern is:

**evidence → draft → evaluate → revise → stop**

The evaluator returns visible criteria. The revision step receives that feedback as input.

We also keep a hard revision budget.

In [ ]:
def draft_answer(question, docs):
    context = format_context(docs)

    prompt = f'''
Answer using only the evidence below.
Cite source IDs such as [KB001].

EVIDENCE:
{context}

QUESTION:
{question}
'''

    return local_generate(
        prompt,
        max_new_tokens=220,
    )


def evaluate_draft(
    question,
    docs,
    draft,
):
    source_ids = [
        d.metadata.get('source_id', '')
        for d in docs
    ]

    has_evidence = bool(docs)
    has_citation = any(
        f'[{source_id}]' in draft
        for source_id in source_ids
        if source_id
    )

    long_enough = len(draft.split()) >= 12

    passed = (
        has_evidence
        and has_citation
        and long_enough
    )

    feedback = []

    if not has_evidence:
        feedback.append(
            'No evidence was supplied.'
        )

    if not has_citation:
        feedback.append(
            'Add at least one valid source citation.'
        )

    if not long_enough:
        feedback.append(
            'Explain the answer a little more clearly.'
        )

    return {
        'passed': passed,
        'has_evidence': has_evidence,
        'has_citation': has_citation,
        'long_enough': long_enough,
        'feedback': feedback,
    }

In [ ]:
def revise_draft(
    question,
    docs,
    draft,
    evaluation,
):
    context = format_context(docs)
    feedback = '\n'.join(
        f'- {item}'
        for item in evaluation['feedback']
    )

    prompt = f'''
Revise the draft using the feedback.
Use only the supplied evidence.
Keep valid source IDs in the answer.

EVIDENCE:
{context}

QUESTION:
{question}

DRAFT:
{draft}

FEEDBACK:
{feedback}
'''

    return local_generate(
        prompt,
        max_new_tokens=240,
    )

In [ ]:
def run_refinement_loop(
    question,
    max_revisions=1,
):
    docs = retriever.invoke(question)
    draft = draft_answer(question, docs)

    history = []

    for revision in range(
        max_revisions + 1
    ):
        evaluation = evaluate_draft(
            question,
            docs,
            draft,
        )

        history.append({
            'revision': revision,
            'draft': draft,
            'evaluation': evaluation,
        })

        if evaluation['passed']:
            return {
                'answer': draft,
                'passed': True,
                'history': history,
            }

        if revision >= max_revisions:
            break

        draft = revise_draft(
            question,
            docs,
            draft,
            evaluation,
        )

    return {
        'answer': draft,
        'passed': False,
        'history': history,
    }

In [ ]:
refined_result = run_refinement_loop(
    "What is the standard return window?",
    max_revisions=1,
)

print(refined_result['answer'])
print('Passed evaluator:', refined_result['passed'])

for item in refined_result['history']:
    print('\nRevision:', item['revision'])
    print(item['evaluation'])

## 17. Optional: compare with LangChain `create_agent`

The notebook is already fully usable without an API key.

This optional section shows a high-level tool-using LangChain agent. The important project-design idea is that the agent receives an **approved tool set from the registry**, not every possible application capability.

The provider example remains optional because native tool calling depends on the selected chat-model integration.

In [ ]:
import getpass
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent

provider_agent = None

if os.getenv('OPENAI_API_KEY'):
    provider_llm = ChatOpenAI(
        model='gpt-5-mini'
    )

    approved_tools = tools_for_project(
        needs_order=True,
        allow_web=True,
    )

    provider_agent = create_agent(
        model=provider_llm,
        tools=approved_tools,
        system_prompt=(
            'Use search_support_knowledge for internal policies. '
            'Use get_order for exact order facts. '
            'Use DuckDuckGo only for explicitly public/current information. '
            'Never let public web results override internal company policy.'
        ),
    )

print(
    'Optional provider agent ready:',
    provider_agent is not None,
)
print(
    'The local-model LangGraph workflow below does not require an API key.'
)

In [ ]:
if provider_agent is not None:
    result = provider_agent.invoke({
        'messages': [
            {
                'role': 'user',
                'content': 'What is the standard return window in our knowledge base?',
            }
        ]
    })
    print(result['messages'][-1].content)
else:
    print('Optional provider example skipped. No API key is required for the rest of the notebook.')

In [ ]:
if provider_agent is not None:
    result = provider_agent.invoke({
        'messages': [
            {
                'role': 'user',
                'content': 'Search the public web for the current LangGraph documentation and explain StateGraph.',
            }
        ]
    })
    print(result['messages'][-1].content)
else:
    print('Continue to the local-model LangGraph agent below.')

## 18. Build the same bounded workflow with LangGraph + the local LLM

Now we translate the controller into graph state, nodes, and conditional edges.

Key design rules:

- internal evidence: retrieve → grade → generate or rewrite,
- web search: only when explicitly allowed,
- stop safely when the retrieval budget is exhausted,
- stop if a rewrite retrieves the same source set again,
- keep a visible trace of actions and stop reason.

**This section still uses the downloaded local Qwen3 model, so no LLM API key is required.**

In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

MAX_RETRIEVAL_STEPS = 3


class RAGState(TypedDict):
    question: str
    query: str
    route: str
    documents: list[dict]
    web_results: str
    score: float
    steps: int
    answer: str
    allow_web: bool
    previous_source_ids: list[str]
    current_source_ids: list[str]
    stop_reason: str
    trace: list[dict]

In [ ]:
def serialize_docs(docs):
    return [
        {
            'text': d.page_content,
            'source_id': d.metadata.get(
                'source_id',
                'unknown',
            ),
            'category': d.metadata.get(
                'category',
                '',
            ),
            'source_url': d.metadata.get(
                'source_url',
                '',
            ),
        }
        for d in docs
    ]


def route_node(state: RAGState):
    refined = refine_input(
        state['question']
    )

    wants_web = (
        state.get('allow_web', False)
        and refined['needs_web']
    )

    route = (
        'web'
        if wants_web
        else 'retrieve'
    )

    return {
        'route': route,
        'query': refined['refined_query'],
        'trace': state['trace'] + [
            {
                'action': 'route',
                'route': route,
            }
        ],
    }


def route_after_start(state: RAGState):
    return state['route']

In [ ]:
def retrieve_node(state: RAGState):
    docs = retriever.invoke(
        state['query']
    )

    source_ids = [
        d.metadata.get(
            'source_id',
            'unknown',
        )
        for d in docs
    ]

    return {
        'documents': serialize_docs(docs),
        'steps': state['steps'] + 1,
        'previous_source_ids': (
            state['current_source_ids']
        ),
        'current_source_ids': source_ids,
        'trace': state['trace'] + [
            {
                'action': 'retrieve',
                'query': state['query'],
                'sources': source_ids,
            }
        ],
    }


def grade_node(state: RAGState):
    docs = [
        Document(
            page_content=d['text'],
            metadata={
                'source_id': d['source_id'],
                'category': d['category'],
                'source_url': d['source_url'],
            },
        )
        for d in state['documents']
    ]

    score = evidence_score(
        state['question'],
        docs,
    )

    return {
        'score': score,
        'trace': state['trace'] + [
            {
                'action': 'grade',
                'score': round(score, 3),
            }
        ],
    }


def route_after_grade(state: RAGState):
    if state['score'] >= 0.25:
        return 'generate'

    if (
        state['previous_source_ids']
        and state['previous_source_ids']
        == state['current_source_ids']
    ):
        return 'stop'

    if state['steps'] >= MAX_RETRIEVAL_STEPS:
        return 'stop'

    return 'rewrite'

In [ ]:
def rewrite_node(state: RAGState):
    new_query = rewrite_query(
        state['question']
    )

    return {
        'query': new_query,
        'trace': state['trace'] + [
            {
                'action': 'rewrite',
                'query': new_query,
            }
        ],
    }


def web_node(state: RAGState):
    try:
        text = web_search.invoke(
            state['query']
        )
    except Exception as exc:
        text = (
            'Web search unavailable: '
            + str(exc)
        )

    return {
        'web_results': str(text),
        'steps': state['steps'] + 1,
        'trace': state['trace'] + [
            {
                'action': 'web_search',
            }
        ],
    }


def generate_node(state: RAGState):
    internal_context = '\n\n'.join(
        f"[{d['source_id']}] {d['text']}"
        for d in state.get(
            'documents',
            [],
        )
    )

    web_context = state.get(
        'web_results',
        '',
    )

    context = internal_context

    if web_context:
        context += (
            '\n\nPUBLIC WEB RESULTS:\n'
            + web_context
        )

    if not context.strip():
        return {
            'answer': (
                'I could not find enough evidence '
                'to answer.'
            ),
            'route': 'stop',
            'stop_reason': 'no_evidence',
        }

    prompt = f'''
Answer from the retrieved evidence.
Keep internal policy evidence and public web evidence clearly separated.
Do not treat public web results as internal company policy.
If evidence is insufficient, say so.

EVIDENCE:
{context}

QUESTION:
{state['question']}
'''

    answer = local_generate(
        prompt,
        max_new_tokens=260,
    )

    return {
        'answer': answer,
        'route': 'generate',
        'stop_reason': 'answer_generated',
        'trace': state['trace'] + [
            {
                'action': 'generate',
            }
        ],
    }


def stop_node(state: RAGState):
    if (
        state['previous_source_ids']
        and state['previous_source_ids']
        == state['current_source_ids']
    ):
        reason = 'no_progress_same_evidence'
    else:
        reason = 'max_retrieval_steps'

    return {
        'route': 'stop',
        'stop_reason': reason,
        'answer': (
            'I could not find enough reliable evidence '
            'to answer this question.'
        ),
        'trace': state['trace'] + [
            {
                'action': 'stop',
                'reason': reason,
            }
        ],
    }

In [ ]:
builder = StateGraph(RAGState)

builder.add_node('route', route_node)
builder.add_node('retrieve', retrieve_node)
builder.add_node('grade', grade_node)
builder.add_node('rewrite', rewrite_node)
builder.add_node('web', web_node)
builder.add_node('generate', generate_node)
builder.add_node('stop', stop_node)

builder.add_edge(START, 'route')

builder.add_conditional_edges(
    'route',
    route_after_start,
    {
        'retrieve': 'retrieve',
        'web': 'web',
    },
)

builder.add_edge('retrieve', 'grade')

builder.add_conditional_edges(
    'grade',
    route_after_grade,
    {
        'generate': 'generate',
        'rewrite': 'rewrite',
        'stop': 'stop',
    },
)

builder.add_edge('rewrite', 'retrieve')
builder.add_edge('web', 'generate')
builder.add_edge('generate', END)
builder.add_edge('stop', END)

graph = builder.compile()
print('LangGraph compiled.')

### Inspect the graph structure

`draw_mermaid()` gives a text representation of the graph. You can paste the output into a Mermaid renderer, or use it as a quick way to inspect nodes and edges.

In [ ]:
print(graph.get_graph().draw_mermaid())

### Run the graph with an internal knowledge-base question

In [ ]:
internal_state = {
    'question': 'Can a standard item be returned 20 days after delivery?',
    'query': '',
    'route': '',
    'documents': [],
    'web_results': '',
    'score': 0.0,
    'steps': 0,
    'answer': '',
    'allow_web': False,
    'previous_source_ids': [],
    'current_source_ids': [],
    'stop_reason': '',
    'trace': [],
}

internal_result = graph.invoke(
    internal_state,
    {'recursion_limit': 10},
)

print(internal_result['answer'])
print('\nFinal route:', internal_result['route'])
print('Stop reason:', internal_result['stop_reason'])
print('Steps:', internal_result['steps'])
print('Score:', round(internal_result['score'], 3))
print('Sources:', internal_result['current_source_ids'])

pd.DataFrame(internal_result['trace'])

### Run the graph with an explicitly external/current question

This example allows the web route. The graph should not use public web search unless `allow_web=True` and the wording signals that current/external information is wanted.

In [ ]:
web_state = {
    'question': 'Search the public web for the current LangGraph documentation and explain StateGraph.',
    'query': '',
    'route': '',
    'documents': [],
    'web_results': '',
    'score': 0.0,
    'steps': 0,
    'answer': '',
    'allow_web': True,
    'previous_source_ids': [],
    'current_source_ids': [],
    'stop_reason': '',
    'trace': [],
}

web_graph_result = graph.invoke(
    web_state,
    {'recursion_limit': 10},
)

print(web_graph_result['answer'])
print('\nFinal route:', web_graph_result['route'])
print('Stop reason:', web_graph_result['stop_reason'])
print('Steps:', web_graph_result['steps'])

pd.DataFrame(web_graph_result['trace'])

## 19. Optional local deployment: GGUF / llama.cpp / Ollama

The Transformers model above is easiest in Colab. For a laptop or desktop, a **GGUF** file is often convenient because it can run with llama.cpp-compatible applications.

Official lightweight option:

- Repository: `Qwen/Qwen3-0.6B-GGUF`
- `Qwen3-0.6B-Q8_0.gguf` is about 639 MB.
- It can be run through llama.cpp, and the official model page also provides an Ollama command.

Example terminal commands:

```bash
# llama.cpp / llama CLI
llama cli -hf Qwen/Qwen3-0.6B-GGUF:Q8_0

# Ollama
ollama run hf.co/Qwen/Qwen3-0.6B-GGUF:Q8_0
```

This is optional; the notebook itself already runs the Transformers version directly.

## 20. Reload the persisted Chroma collection

Persistence matters because a real application should not rebuild embeddings every time it starts.

In [ ]:
reloaded_store = Chroma(
    collection_name=COLLECTION_NAME,
    persist_directory=CHROMA_DIR,
    embedding_function=embeddings,
)

print('Reloaded chunks:', len(reloaded_store.get()['ids']))

reloaded_docs = reloaded_store.similarity_search(
    'How long does the limited warranty last?',
    k=3,
)

[(d.metadata['source_id'], d.metadata['title']) for d in reloaded_docs]

## 21. Evaluate retrieval with a small test set

Before evaluating final answers, test whether retrieval finds the expected evidence. This makes it easier to identify whether a failure comes from retrieval or generation.

In [ ]:
tests = [
    ('What is the normal return deadline?', 'returns'),
    ('How long does refund processing take?', 'refunds'),
    ('When should shipping damage be reported?', 'damaged_items'),
    ('How long is the limited warranty?', 'warranty'),
    ('How do I get a password reset link?', 'accounts'),
    ('When can I cancel a subscription?', 'subscriptions'),
]

rows = []

for question, expected_category in tests:
    docs = retriever.invoke(question)
    categories = [d.metadata['category'] for d in docs]
    sources = [d.metadata['source_id'] for d in docs]

    rows.append({
        'question': question,
        'expected_category': expected_category,
        'retrieved_categories': ', '.join(categories),
        'sources': ', '.join(sources),
        'hit': expected_category in categories,
    })

evaluation = pd.DataFrame(rows)
evaluation

In [ ]:
print(
    'Retrieval hit rate:',
    f"{evaluation['hit'].mean():.0%}"
)

## 22. Troubleshooting checklist

### Retrieval returns the wrong material
- Inspect the raw chunks first.
- Try a more specific query.
- Adjust `k`.
- Add metadata filters.
- Consider a stronger embedding model or reranker.

### The agent keeps looping
- Log the query, source IDs, score, selected route, and step count.
- Check whether rewriting actually changes the query.
- Check whether the same document IDs keep returning.
- Check whether the grader threshold can realistically pass.
- Keep both an application step budget and LangGraph `recursion_limit`.

### The agent chooses the wrong tool
- Make tool names more specific.
- Improve descriptions with **when to use / when not to use**.
- Reduce the available tool set.
- Separate overlapping tools.
- Use deterministic routing when the distinction is obvious.

### The answer is weak even though evidence is good
- Do not retrieve again automatically.
- Evaluate the draft and revise generation only.
- Check citations, completeness, and formatting separately.

### The agent uses web search for internal policy
- Keep public web and authoritative internal sources separate.
- Require explicit `allow_web=True`.
- Do not let public web evidence override internal policy.

### Chroma duplicates documents after re-running
- Use stable IDs.
- Reuse the existing collection intentionally, or delete the teaching directory before rebuilding.

### DuckDuckGo fails
- Internet access may be unavailable in the runtime.
- Treat search as an external dependency and handle failure explicitly.

## 23. What you built

You now have the complete no-API-key path:

**CSV → Documents → chunks → embeddings → Chroma → retriever → local Qwen3 → classic RAG → managed tools → DuckDuckGo → input refinement → visible plan → bounded Agentic RAG → evaluator/revision loop → LangGraph with safe stop → persistence → evaluation**

### Project-design lessons

- Choose the simplest architecture that solves the task.
- Select tools from the information need.
- Keep tool contracts narrow and permissions outside the model.
- Preserve the original user intent while refining search/tool input.
- Reuse visible artifacts such as plan, evidence, draft, critique, and trace.
- Correct only the layer that failed.
- Stop when criteria pass or when the iteration budget is exhausted.

The optional provider `create_agent` section is included only for comparison. The main RAG, refinement, and LangGraph workflow runs with the downloaded local model.